# 03. Многоканальный PWM

Многоканальные функции суммируют несколько физических PWM-каналов. В стандартном варианте все каналы читают один и тот же входной сэмпл, но используют разные фазы несущей.

Основные функции:

- `pwm_kind1_multichannel(...)`
- `pwm_kind2_multichannel_latched(...)`

Для показа отдельных каналов в этом ноутбуке используется вспомогательная функция `pwm_kind2_channel_waveforms(...)`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE
elif (HERE / "tutorials" / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE / "tutorials"
else:
    raise RuntimeError("Run this notebook from the repository root or from the tutorials folder")

path_text = str(TUTORIAL_DIR)
if path_text not in sys.path:
    sys.path.insert(0, path_text)

import matplotlib.pyplot as plt
import numpy as np

from tutorial_helpers import (
    configure_plots,
    grouped_fifo_channel_waveforms,
    load_pwm_lab,
    plot_bitstream,
    plot_channel_stack,
    plot_moving_average_reconstruction,
    plot_pwm_carrier_output,
    plot_spectra,
    print_peak_table,
    pwm_kind2_channel_waveforms,
    show_grouped_mapping,
    time_us,
)

pl = load_pwm_lab()
configure_plots()

## Same-phase против phase-interleaved

Same-phase каналы складывают одинаковые импульсы. Phase-interleaved каналы разнесены по фазе несущей, поэтому структура высокочастотной пульсации меняется.

In [ ]:
config = pl.PwmConfig(f_clk=80e6, f_pwm=1e6, resolution_bits=8)
f_signal = 50e3
channels = 4
n_periods = 128

_, x_fifo = pl.sine_samples(freq=f_signal, sample_rate=config.actual_f_pwm, n_samples=n_periods, amplitude=0.85)

traces_same = pwm_kind2_channel_waveforms(
    x_fifo,
    config,
    channels=channels,
    phase_offsets=np.zeros(channels),
)
traces_interleaved = pwm_kind2_channel_waveforms(x_fifo, config, channels=channels)

same_sum = traces_same.mean(axis=0)
interleaved_sum = traces_interleaved.mean(axis=0)

## Временные дорожки каналов

In [ ]:
plot_channel_stack(
    traces_same,
    sample_rate=config.f_clk,
    max_points=8 * config.period_samples,
    summed=same_sum,
    title="Same-phase PWM channels",
);

plot_channel_stack(
    traces_interleaved,
    sample_rate=config.f_clk,
    max_points=8 * config.period_samples,
    summed=interleaved_sum,
    title="Phase-interleaved PWM channels",
);

## Среднее за период

Низкочастотное среднее сохраняет входную огибающую, но высокочастотная часть у суммированных каналов другая.

In [ ]:
avg_same = pl.moving_average_decimate(same_sum, config.period_samples)
avg_interleaved = pl.moving_average_decimate(interleaved_sum, config.period_samples)
t_ms = np.arange(n_periods) / config.actual_f_pwm * 1e3

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.plot(t_ms, x_fifo, "o-", markersize=3, label="input")
ax.plot(t_ms, avg_same, ".-", label="same-phase average")
ax.plot(t_ms, avg_interleaved, "s--", markersize=3, label="interleaved average")
ax.set_title("Period averages for summed channels")
ax.set_xlabel("time, ms")
ax.set_ylabel("normalized amplitude")
ax.legend(loc="upper right");

## Спектр одиночного канала и сумм

In [ ]:
plot_spectra(
    {
        "single channel": traces_interleaved[0],
        "same-phase sum": same_sum,
        "interleaved sum": interleaved_sum,
    },
    sample_rate=config.f_clk,
    f_max=10e6,
    f_scale=1e6,
    f_unit="MHz",
    title="Multi-channel PWM spectra",
);

## Как меняется результат при разном числе каналов

In [ ]:
waveforms = {}
for ch in [1, 2, 4, 8]:
    waveforms[f"{ch} ch"] = pl.pwm_kind2_multichannel_latched(x_fifo, config, channels=ch)

plot_spectra(
    waveforms,
    sample_rate=config.f_clk,
    f_max=10e6,
    f_scale=1e6,
    f_unit="MHz",
    title="Effect of channel count",
);